In [6]:
from acled_model import pre_process_data,train_evaluate_model, mark_conflict_events, create_regional_monthly_baseline
import pandas as pd

In [2]:
all_data = pd.read_csv("../data/all_data.csv")

countries = ["Sudan"]
start_date = "2017-07-01" # TODO validation for 6 month warm up period
end_date = "2024-12-31"

train_start_date = "2018-01-01"
train_end_date = "2022-12-31"

onset_start_date = "2023-01-01"
onset_end_date = "2023-12-31"

active_start_date = "2024-01-01"
active_end_date = "2024-12-31"

## Testing to see which k deviations from the mean works best

In [ ]:
test_ks = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
results = []

for k in test_ks:
    processed_df, predictor_cols = pre_process_data(all_data, k)

    result = train_evaluate_model(all_data, k)
    result["k"] = k
    results.append(result)

print(results)

## Testing to see whether sub events or events work better

In [7]:
def pre_process_data(df, k, event_col = "sub_event_type"):
    df = df.copy()
    df = mark_conflict_events(df)
    df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

    pivot_df = pd.pivot_table(
        df,
        values="event_id_cnty",
        index=["admin1", "year_month"],
        columns=[event_col],
        # columns=["event_type"],
        aggfunc="count",
        fill_value=0,
    ).reset_index()

    pivot_df.columns = (
        pivot_df.columns.str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

    baseline_df = create_regional_monthly_baseline(df,k)

    fatalities_df = (
        df.groupby(["admin1", "year_month"])["fatalities"].sum().reset_index()
    )

    combined_df = pd.merge(
        baseline_df, pivot_df, on=["admin1", "year_month"], how="left")
    combined_df = pd.merge(
        combined_df, fatalities_df, on=["admin1", "year_month"], how="left")

    event_cols = pivot_df.columns.drop(["admin1", "year_month"]).tolist()
    combined_df[event_cols] = combined_df[event_cols].fillna(0)
    combined_df["fatalities"] = combined_df["fatalities"].fillna(0)

    current_event_cols = event_cols + ["fatalities"]
    lagged_event_cols = ["rolling_mean_6m", "rolling_std_6m", "escalation_threshold"]

    combined_df[current_event_cols] = combined_df[current_event_cols].fillna(0)
    combined_df[current_event_cols] = combined_df.groupby("admin1")[current_event_cols].shift(1)

    predictor_cols = current_event_cols + lagged_event_cols
    combined_df[predictor_cols] = combined_df[predictor_cols].fillna(0)

    combined_df = combined_df.rename(columns={"admin1": "region"})
    combined_df = combined_df.sort_values(by=["year_month", "region"]).reset_index(drop=True)
    return combined_df, predictor_cols

In [ ]:
processed_df, predictor_cols = pre_process_data(all_data, 0.5, "sub_event_type")

result = train_evaluate_model(all_data, 0.5)
print("sub_event_type", result)

processed_df, predictor_cols = pre_process_data(all_data, 0.5, "event_type")

result = train_evaluate_model(all_data, 0.5)
print("event_type", result)

sub_event_type {'optimal_threshold': '0.0880', 'onset_aupr': '0.3847', 'onset_precision_class1': '0.3825', 'onset_recall_class1': '0.9765', 'onset_f1_class1': '0.5497', 'active_aupr': '0.3989', 'active_precision_class1': '0.3991', 'active_recall_class1': '0.9884', 'active_f1_class1': '0.5686'}
